In [8]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from datetime import datetime, timedelta

API_KEY = '1e86a5a049014d2f8dbce5fcd4c3e821'

# 1. 전체 공연 목록(요약) 가져오기
def fetch_all_list():
    url = "http://www.kopis.or.kr/openApi/restful/pblprfr"
    performance_list = []
    page = 1
    
    # 실행되는 시점의 날짜 자동 계산
    today = datetime.now()
    stdate_str = today.strftime('%Y%m%d') 
    eddate_str = (today + timedelta(days=90)).strftime('%Y%m%d') 
    
    print(f"=== [1단계] {stdate_str} 기준 최신 데이터 수집 시작 ===")
    
    while True:
        params = {
            'service': API_KEY,
            'stdate': stdate_str,   
            'eddate': eddate_str,   
            'cpage': str(page),
            'rows': '100',          
            'shcate': 'CCCA'        
        }
        
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            print(f"API 요청 중단 (상태 코드: {response.status_code}) - 마지막 페이지 도달 추정")
            break
            
        root = ET.fromstring(response.content)
        db_elements = root.findall('db')
        
        root = ET.fromstring(response.content)
        db_elements = root.findall('db')
        
        # 데이터가 없거나, '정상적인 공연 ID'가 없는 에러 메시지일 경우 즉시 탈출
        if not db_elements or db_elements[0].findtext('mt20id') is None:
            break
            
        for db in db_elements:
            performance = {
                '공연ID': db.findtext('mt20id'),
                '공연명': db.findtext('prfnm'),
                '시작일': db.findtext('prfpdfrom'),
                '종료일': db.findtext('prfpdto'),
                '공연장소': db.findtext('fcltynm'),
                '장르': db.findtext('genrenm'),
                '포스터': db.findtext('poster').replace('http://', 'https://') if db.findtext('poster') else '',
                '상태': db.findtext('prfstate')
            }
            performance_list.append(performance)
            
        print(f"목록 수집 중... {page}페이지 완료 (누적: {len(performance_list)}건)")
        
        # 가져온 데이터가 100개(요청 건수) 미만이라면? 여기가 마지막 페이지라는 뜻! 무한 루프 종료
        if len(db_elements) < 100:
            print("더 이상 수집할 데이터가 없어 목록 수집을 종료합니다.")
            break
            
        page += 1
        time.sleep(0.5)
            
        print(f"목록 수집 중... {page}페이지 완료 (누적: {len(performance_list)}건)")
        page += 1
        time.sleep(0.5) 
        
    return pd.DataFrame(performance_list)

# 주소에서 '서울', '경기' 등 깔끔한 지역명만 뽑아내는 함수
def get_simple_region(address):
    # 주소가 아예 없거나, 스페이스바 공백(" ")만 있는 경우를 완벽하게 차단
    if not address or not str(address).strip():
        return "기타"
    
    # 주소의 첫 번째 단어 추출 (안전하게 문자열로 변환 후 처리)
    sido = str(address).split()[0] 
    
    # 깔끔한 지역명으로 매핑
    mapping = {
        "서울특별시": "서울", "서울시": "서울", "서울": "서울",
        "경기도": "경기", "경기": "경기",
        "인천광역시": "인천", "인천시": "인천",
        "부산광역시": "부산", "부산시": "부산",
        "대구광역시": "대구", "대구시": "대구",
        "광주광역시": "광주", "광주시": "광주",
        "대전광역시": "대전", "대전시": "대전",
        "울산광역시": "울산", "울산시": "울산",
        "세종특별자치시": "세종", "세종시": "세종",
        "강원특별자치도": "강원", "강원도": "강원",
        "충청북도": "충북", "충청남도": "충남",
        "전북특별자치도": "전북", "전라북도": "전북",
        "전라남도": "전남", "경상북도": "경북", "경상남도": "경남",
        "제주특별자치도": "제주", "제주도": "제주"
    }
    return mapping.get(sido, sido)


# 2. 개별 공연 상세 정보
def fetch_all_details(id_list):
    detail_list = []
    total = len(id_list)
    
    # 똑같은 공연장 주소를 매번 조회하지 않기 위해 저장해두는 기억 장치(캐시)
    venue_cache = {} 
    
    print(f"\n=== [2단계] 총 {total}개 공연 상세 정보 및 지역 데이터 수집 시작 ===")
    
    for i, mt20id in enumerate(id_list):
        url = f"http://www.kopis.or.kr/openApi/restful/pblprfr/{mt20id}"
        params = {'service': API_KEY}
        
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            db = root.find('db')
            
            if db is not None:
                # ----------------------------------------------------
                # [지역 정보 추출 로직]
                mt10id = db.findtext('mt10id') # 공연시설 ID
                region = "지역미상"
                
                if mt10id:
                    if mt10id in venue_cache:
                        # 이미 검색해본 장소면 기억해둔 지역명 바로 꺼내 쓰기
                        region = venue_cache[mt10id]
                    else:
                        # 처음 보는 장소면 시설 API 호출해서 주소 알아내기
                        fac_url = f"http://www.kopis.or.kr/openApi/restful/prfplc/{mt10id}"
                        fac_res = requests.get(fac_url, params={'service': API_KEY})
                        if fac_res.status_code == 200:
                            fac_root = ET.fromstring(fac_res.content)
                            fac_db = fac_root.find('db')
                            if fac_db is not None:
                                adres = fac_db.findtext('adres')
                                region = get_simple_region(adres)
                        
                        # 다음번을 위해 딕셔너리에 기억해두기
                        venue_cache[mt10id] = region
                        time.sleep(0.1) # 시설 API 서버 보호를 위한 찰나의 휴식
                # ----------------------------------------------------

                # 기존 텍스트, 이미지, 예매처 추출 로직 (동일)
                sty_text = db.findtext('sty')
                if not sty_text or sty_text.strip() == "":
                    sty_text = "상세 프로그램은 하단 상세 이미지를 참조해주세요."
                
                image_urls = []
                styurls_element = db.find('styurls')
                if styurls_element is not None:
                    for styurl in styurls_element.findall('styurl'):
                        if styurl.text:
                            image_urls.append(styurl.text.replace('http://', 'https://'))
                joined_image_urls = ", ".join(image_urls)
                
                booking_links = []
                relates_element = db.find('relates')
                if relates_element is not None:
                    for relate in relates_element.findall('relate'):
                        relatenm = relate.findtext('relatenm')
                        relateurl = relate.findtext('relateurl')
                        if relatenm and relateurl:
                            booking_links.append(f"{relatenm}({relateurl})")
                joined_booking_links = ", ".join(booking_links) if booking_links else "예매처 정보 없음"
                
                # 데이터 병합
                detail = {
                    '공연ID': mt20id,
                    '출연진': db.findtext('prfcast'),
                    '런타임': db.findtext('prfruntime'),
                    '관람연령': db.findtext('prfage'),
                    '티켓가격': db.findtext('pcseguidance'),
                    '소개글_프로그램': sty_text,
                    '상세이미지_URL': joined_image_urls,
                    '예매처_링크': joined_booking_links,
                    '지역': region 
                }
                detail_list.append(detail)
                
        time.sleep(0.3) 
        
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"상세 정보 수집 진행률: {i + 1} / {total} 완료")
            
    return pd.DataFrame(detail_list)

# 3. 메인 실행 흐름
if __name__ == "__main__":
    # 1단계 실행: 목록 요약본 가져오기
    list_df = fetch_all_list()
    
    if not list_df.empty:
        # 2단계 실행: 목록에서 ID만 뽑아서 상세 정보 가져오기
        all_ids = list_df['공연ID'].tolist()
        detail_df = fetch_all_details(all_ids)
        
        # 3단계: 두 데이터를 '공연ID' 기준으로 하나로 합치기
        print("\n=== [3단계] 데이터 병합 및 CSV 저장 ===")
        final_df = pd.merge(list_df, detail_df, on='공연ID', how='left')
        
        # 최종 결과를 CSV 파일로 저장
        file_name = 'kopis_classic_final_data.csv'
        final_df.to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f" 모든 수집이 완료되었습니다! 파일명: '{file_name}'")
    else:
        print("수집된 목록 데이터가 없습니다. 날짜나 조건을 다시 확인해주세요.")

=== [1단계] 20260515 기준 최신 데이터 수집 시작 ===
목록 수집 중... 1페이지 완료 (누적: 100건)
목록 수집 중... 2페이지 완료 (누적: 100건)
목록 수집 중... 3페이지 완료 (누적: 200건)
목록 수집 중... 4페이지 완료 (누적: 200건)
목록 수집 중... 5페이지 완료 (누적: 300건)
목록 수집 중... 6페이지 완료 (누적: 300건)
목록 수집 중... 7페이지 완료 (누적: 400건)
목록 수집 중... 8페이지 완료 (누적: 400건)


KeyboardInterrupt: 